In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os, time
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from PIL import Image
from torchvision import datasets, transforms
import random
from torch.utils.data import DataLoader, Subset, Dataset

def generate_predictions(input_dir, output_dir):

    DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    BATCH_SIZE = 32
    IMG_SIZE   = 128
    EPOCHS     = 8
    PIN        = torch.cuda.is_available()
    print("Device:", DEVICE, flush=True)

    TRAIN_PATH = os.path.join(input_dir, "train", "train")
    TEST_PATH  = os.path.join(input_dir, "test", "test")

    # -------- TRANSFORMS (UNCHANGED) --------
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=6, translate=(0.08, 0.08), scale=(0.95, 1.05)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        transforms.RandomErasing(p=0.2)
    ])

    test_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])

    # -------- DATA (UNCHANGED) --------
    full_train = datasets.ImageFolder(TRAIN_PATH, transform=train_transform)
    full_val   = datasets.ImageFolder(TRAIN_PATH, transform=test_transform)

    n = len(full_train)
    indices = list(range(n))

    random.seed(42)
    random.shuffle(indices)

    train_size = int(0.85 * n)
    train_indices = indices[:train_size]
    val_indices   = indices[train_size:]

    train_data = Subset(full_train, train_indices)
    val_data   = Subset(full_val, val_indices)

    train_loader = DataLoader(train_data, BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_data, BATCH_SIZE, shuffle=False, num_workers=0)

    print(f"Train:{len(train_data)} Val:{len(val_data)} Batches:{len(train_loader)}", flush=True)

    # -------- MODEL (UNCHANGED) --------
    class ResBlock(nn.Module):
        def __init__(self, ch):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(ch, ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(ch), nn.ReLU(inplace=True),
                nn.Conv2d(ch, ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(ch),
            )
        def forward(self, x): return torch.relu(self.net(x) + x)

    class SEBlock(nn.Module):
        def __init__(self, ch, r=16):
            super().__init__()
            self.se = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Linear(ch, ch // r),
                nn.ReLU(),
                nn.Linear(ch // r, ch),
                nn.Sigmoid()
            )
        def forward(self, x):
            return x * self.se(x).view(x.size(0), -1, 1, 1)

    class FastCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(3,32,3,padding=1,bias=False),
                nn.BatchNorm2d(32), nn.ReLU(inplace=True),

                nn.Conv2d(32,64,3,padding=1,bias=False),
                nn.BatchNorm2d(64), nn.ReLU(inplace=True),
                ResBlock(64),
                nn.MaxPool2d(2),

                nn.Conv2d(64,128,3,padding=1,bias=False),
                nn.BatchNorm2d(128), nn.ReLU(inplace=True),
                ResBlock(128),
                nn.MaxPool2d(2),

                nn.Conv2d(128,256,3,padding=1,bias=False),
                nn.BatchNorm2d(256), nn.ReLU(inplace=True),
                ResBlock(256),
                SEBlock(256),

                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Dropout(0.2)
            )
            self.fc = nn.Linear(256, 2)

        def forward(self, x):
            return self.fc(self.net(x))

    model     = FastCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=1e-3,
        steps_per_epoch=len(train_loader), epochs=EPOCHS
    )

    best_acc, best_state = 0.0, None

    # -------- TRAINING (UNCHANGED) --------
    for epoch in range(1, EPOCHS+1):
        model.train()
        t0 = time.time()
        loss_sum = cor = tot = 0
        print(f"\n[Epoch {epoch}/{EPOCHS}]", flush=True)

        for i, (imgs, lbls) in enumerate(train_loader, 1):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward(); optimizer.step(); scheduler.step()

            loss_sum += loss.item()
            cor += (out.argmax(1) == lbls).sum().item()
            tot += lbls.size(0)

            if i % 20 == 0 or i == len(train_loader):
                print(f"  [{i}/{len(train_loader)}] loss={loss_sum/i:.4f} acc={cor/tot:.4f}", flush=True)

        model.eval(); vc = vt = 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                vc += (model(imgs).argmax(1) == lbls).sum().item()
                vt += lbls.size(0)

        val_acc = vc/vt
        print(f"[Epoch {epoch}] train={cor/tot:.4f} val={val_acc:.4f} ({time.time()-t0:.0f}s)", flush=True)

        if val_acc > best_acc and epoch >= 3:
            best_acc  = val_acc
            best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
            print(f"  ✓ Best: {best_acc:.4f}", flush=True)

    model.load_state_dict(best_state)

    # -------- TEST (UNCHANGED) --------
    class TestDataset(Dataset):
        def __init__(self, folder, transform=None):
            self.folder    = folder
            self.transform = transform
            self.images    = sorted(os.listdir(folder),
                key=lambda x: int(os.path.splitext(x)[0]) if os.path.splitext(x)[0].isdigit() else x)

        def __len__(self): return len(self.images)

        def __getitem__(self, idx):
            name = self.images[idx]
            img  = Image.open(os.path.join(self.folder, name)).convert("RGB")
            if self.transform: img = self.transform(img)
            return img, name

    test_ds = TestDataset(TEST_PATH, test_transform)
    test_ld = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=0)

    print(f"\n[INFO] Test images: {len(test_ds)}", flush=True)

    model.eval()
    predictions = []
    with torch.no_grad():
        for imgs, names in test_ld:
            preds = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
            for name, pred in zip(names, preds):
                predictions.append((name, int(pred)))

    df = pd.DataFrame(predictions, columns=["ID","Label"])

    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "submission.csv")
    df.to_csv(output_path, index=False)

    print("✅ Done! submission.csv saved", flush=True)
    print(df.head())

    return output_path


if __name__ == "__main__":
    input_dir = input("enter input directory:").strip()
    output_dir = input("enter output directory").strip()
    generate_predictions(input_dir, output_dir)